In [1]:
import os
import numpy as np
import pandas as pd

# ===== Parámetros =====
data_folder = 'data/BACI_HS92_V202601'
YEARS = [2021, 2022, 2023]                  # años que se promedian
MIN_EXPORT = 100                            # miles USD/año para contar como exportador
MIN_COUNTRY_TOTAL = 1_000_000               # miles USD: filtro de país (USD 1.000 M)
BASKET_PREFIXES = ['8401', '2844', '8109']  # canasta nuclear (HS92)

# ===== 1. Diccionarios =====
df_countries = pd.read_csv(os.path.join(data_folder, 'country_codes_V202601.csv'))
df_products = pd.read_csv(os.path.join(data_folder, 'product_codes_HS92_V202601.csv'))
df_products['code'] = pd.to_numeric(df_products['code'], errors='coerce')
df_products = df_products.dropna(subset=['code'])
df_products['code'] = df_products['code'].astype(int)

country_map = dict(zip(df_countries['country_code'], df_countries['country_iso3']))
product_map = dict(zip(df_products['code'], df_products['description']))

# ===== 2. Exportación anual promedio =====
frames = []
for y in YEARS:
    f = os.path.join(data_folder, f'BACI_HS92_Y{y}_V202601.csv')
    d = pd.read_csv(f, usecols=['i', 'k', 'v'])   # i=país, k=producto, v=miles USD
    frames.append(d.groupby(['i', 'k'])['v'].sum())

df_trade = (pd.concat(frames).groupby(level=[0, 1]).sum() / len(YEARS)).reset_index()
df_trade['country'] = df_trade['i'].map(country_map)
df_trade = df_trade.dropna(subset=['country'])

# ===== 3. Matriz país x producto =====
X = df_trade.pivot_table(index='country', columns='k', values='v',
                         aggfunc='sum', fill_value=0.0)
X = X.loc[X.sum(axis=1) > MIN_COUNTRY_TOTAL]
X = X.loc[:, X.sum(axis=0) > 0]
print(f'Matriz: {X.shape[0]} países x {X.shape[1]} productos')

# ===== 4. RCA y matriz binaria M =====
X_vals = X.values
RCA = (X_vals / X_vals.sum(axis=1, keepdims=True)) / (X_vals.sum(axis=0, keepdims=True) / X_vals.sum())
M = ((RCA >= 1.0) & (X_vals >= MIN_EXPORT)).astype(int)

# ===== 5. Proximidad Phi =====
C = M.T @ M                                   # países que exportan ambos productos
ubiq = M.sum(axis=0)                          # exportadores por producto
max_ubiq = np.maximum(ubiq[:, None], ubiq[None, :])
Phi = np.divide(C, max_ubiq, out=np.zeros(C.shape, dtype=float), where=max_ubiq != 0)
np.fill_diagonal(Phi, 0.0)

# ===== 6. Canasta y densidad =====
cols = list(X.columns)
basket = [c for c in cols if str(int(c)).zfill(6).startswith(tuple(BASKET_PREFIXES))]
idxs = [cols.index(c) for c in basket]

print('\nCanasta nuclear:')
for c, j in zip(basket, idxs):
    print(f'  {c} | exportadores: {ubiq[j]:3d} | {str(product_map.get(c, "?"))[:70]}')

den = Phi[:, idxs].sum(axis=0).astype(float)
den[den == 0] = np.nan
Omega = pd.DataFrame((M @ Phi[:, idxs]) / den, index=X.index, columns=basket)
omega_basket = Omega.mean(axis=1)             # densidad promedio hacia la canasta

print('\nTop 10 por densidad:')
print(omega_basket.sort_values(ascending=False).head(10).round(3))

# ===== 7. Colombia =====
if 'COL' in X.index:
    i_col = list(X.index).index('COL')
    val = omega_basket['COL']
    rank = int(omega_basket.rank(ascending=False)['COL'])
    print(f'\nDensidad COL: {val:.4f} | puesto {rank} de {len(omega_basket)}')

    # Línea base: COL hacia cualquier producto
    den_all = Phi.sum(axis=0).astype(float)
    den_all[den_all == 0] = np.nan
    omega_col_all = (M[i_col] @ Phi) / den_all
    pct = np.nanmean(omega_col_all < val) * 100
    print(f'Promedio COL (cualquier producto): {np.nanmean(omega_col_all):.4f} | percentil {pct:.0f}')

    print('\nDensidad COL por código:')
    print(Omega.loc['COL'].round(4).to_string())
else:
    print("No se encontró 'COL'.")

Matriz: 164 países x 4656 productos

Canasta nuclear:
  284410 | exportadores:   9 | Uranium: natural uranium and its compounds, alloys, dispersions (inclu
  284420 | exportadores:   8 | Uranium: enriched in U235, plutonium, their compounds, alloys dispersi
  284430 | exportadores:  10 | Uranium: depleted in U235, thorium, their compounds, alloys, dispersio
  284440 | exportadores:  16 | Radioactive elements, isotopes, compounds, n.e.s. in heading no. 2844 
  284450 | exportadores:   6 | Spent (irradiated) fuel elements (cartridges) of nuclear reactors
  810910 | exportadores:   8 | Zirconium: unwrought, waste and scrap, powders
  810990 | exportadores:   8 | Zirconium: other than unwrought, n.e.s. in heading no. 8109
  840110 | exportadores:   3 | Nuclear reactors
  840120 | exportadores:   7 | Machinery and apparatus: for isotopic separation, and parts thereof
  840130 | exportadores:  10 | Fuel elements (cartridges): non-irradiated
  840140 | exportadores:   8 | Nuclear reactors: pa

In [2]:
# Chequeo de cordura: puesto de países con programa nuclear (diapo 17)
paises = ['RUS', 'KOR', 'CAN', 'IND', 'ARG', 'BRA', 'COL']
rank_all = omega_basket.rank(ascending=False)

tabla = pd.DataFrame({
    'Puesto': rank_all.reindex(paises).astype('Int64'),
    'Densidad': omega_basket.reindex(paises).round(3),
}).dropna(how='all')

print(f'Total de países: {len(omega_basket)}')
print(tabla)

Total de países: 164
         Puesto  Densidad
country                  
RUS          44     0.131
KOR          25     0.201
CAN          23     0.208
IND          12     0.310
ARG          59     0.086
BRA          53     0.110
COL          73     0.072


In [3]:
for pref in ['9030', '9022', '9027', '3006']:
    print(pref, [(c, str(product_map.get(c))[:60]) for c in cols if str(int(c)).zfill(6).startswith(pref)])

9030 [(903010, 'Instruments and apparatus: for measuring or detecting ionisi'), (903020, 'Cathode-ray oscilloscopes and cathode-ray oscillographs'), (903031, 'Multimeters: for measuring or checking voltage, current, res'), (903039, 'Instruments and apparatus: for measuring or checking voltage'), (903040, 'Instruments and apparatus: specially designed for telecommun'), (903081, 'Instruments and apparatus: with a recording device, speciall'), (903089, 'Instruments and apparatus: for measuring or detecting alpha,'), (903090, 'Instruments, apparatus for measuring, checking electrical qu')]
9022 [(902211, 'Apparatus based on the use of x-rays, including radiography '), (902219, 'Apparatus based on the use of x-rays, including radiography '), (902221, 'Apparatus based on the use of alpha, beta or gamma radiation'), (902229, 'Apparatus based on the use of alpha, beta or gamma radiation'), (902230, 'X-ray tubes'), (902290, 'Apparatus based on use of x-rays and similar: parts and acce')]
9027 [

In [4]:
nichos = [903010, 902211, 902219, 902221, 902229, 902230]
nichos = [k for k in nichos if k in cols]      # solo los que existen

filas = []
for k in nichos:
    j = cols.index(k)
    den_j = Phi[:, j].sum()
    om = pd.Series((M @ Phi[:, j]) / den_j, index=X.index)   # densidad de cada país hacia k
    filas.append({
        'Código': k,
        'Exportadores': int(ubiq[j]),
        'COL exporta': 'SÍ' if M[list(X.index).index('COL'), j] else 'NO',
        'Densidad COL': round(om['COL'], 3),
        'Puesto COL': int(om.rank(ascending=False)['COL']),
        'Densidad media': round(om.mean(), 3),
    })
print(pd.DataFrame(filas).to_string(index=False))

 Código  Exportadores COL exporta  Densidad COL  Puesto COL  Densidad media
 903010            20          NO         0.090          74           0.118
 902211             9          NO         0.064          68           0.094
 902219            16          NO         0.077          68           0.108
 902221            10          NO         0.070          71           0.102
 902229            11          NO         0.069          71           0.097
 902230             9          NO         0.063          68           0.093


In [5]:
# Modelo nulo: ¿cuánta densidad tendría cada país SOLO por ser diversificado?
# Se barajan los enlaces de M conservando:
#   - diversidad de cada país (cuántos productos exporta)  -> suma por fila
#   - ubicuidad de cada producto (cuántos países lo exportan) -> suma por columna
# Algoritmo "curveball": dos países intercambian al azar los productos que no comparten.
rng = np.random.default_rng(2026)
N_MUESTRAS = 30          # redes aleatorias
PASOS = 5 * M.shape[0]   # intercambios por red

def omega_canasta(Mx):
    ub = Mx.sum(axis=0)
    C = Mx.T @ Mx[:, idxs]                                   # coexportadores con la canasta
    mx = np.maximum(ub[:, None], ub[idxs][None, :])
    P = np.divide(C, mx, out=np.zeros(C.shape), where=mx != 0)
    P[idxs, range(len(idxs))] = 0.0                          # sin auto-proximidad
    d = P.sum(axis=0); d[d == 0] = np.nan
    return ((Mx @ P) / d).mean(axis=1)                       # promedio sobre la canasta

filas = [set(np.flatnonzero(M[i])) for i in range(M.shape[0])]
nulos = []
for s in range(N_MUESTRAS):
    for _ in range(PASOS):
        a, b = rng.choice(len(filas), 2, replace=False)
        A, B = filas[a], filas[b]
        solo_a, solo_b = list(A - B), list(B - A)
        bolsa = solo_a + solo_b
        rng.shuffle(bolsa)
        comun = A & B
        filas[a] = comun | set(bolsa[:len(solo_a)])
        filas[b] = comun | set(bolsa[len(solo_a):])
    Mn = np.zeros_like(M)
    for i, f in enumerate(filas):
        Mn[i, list(f)] = 1
    nulos.append(omega_canasta(Mn))
    print(f'Red aleatoria {s + 1}/{N_MUESTRAS}', end='\r')

nulos = np.array(nulos)
res = pd.DataFrame({
    'omega_real': omega_basket.values,
    'omega_nulo': nulos.mean(axis=0),
    'sd_nulo': nulos.std(axis=0),
    'diversidad': M.sum(axis=1),
}, index=X.index)
res['exceso'] = res['omega_real'] - res['omega_nulo']
res['z'] = res['exceso'] / res['sd_nulo']
res['puesto_real'] = res['omega_real'].rank(ascending=False).astype(int)
res['puesto_exceso'] = res['exceso'].rank(ascending=False).astype(int)

print(f'\nCorrelación omega real vs nulo: {res["omega_real"].corr(res["omega_nulo"]):.2f}')
print(f'Correlación omega real vs diversidad: {res["omega_real"].corr(res["diversidad"]):.2f}')
print('\nTop 10 por EXCESO (cercanía que no explica la diversificación):')
print(res.sort_values('exceso', ascending=False).head(10)[['omega_real', 'omega_nulo', 'exceso', 'z', 'puesto_real']].round(3))
print('\nPaíses de referencia:')
print(res.loc[['DEU', 'IND', 'CAN', 'KOR', 'RUS', 'BRA', 'ARG', 'COL'],
              ['omega_real', 'omega_nulo', 'exceso', 'z', 'puesto_real', 'puesto_exceso']].round(3))

Red aleatoria 30/30
Correlación omega real vs nulo: 0.97
Correlación omega real vs diversidad: 0.97

Top 10 por EXCESO (cercanía que no explica la diversificación):
         omega_real  omega_nulo  exceso       z  puesto_real
country                                                     
GBR           0.373       0.269   0.104  11.118            7
USA           0.393       0.290   0.103   9.036            6
DEU           0.540       0.436   0.103   7.075            1
FRA           0.477       0.377   0.100   7.467            3
SWE           0.294       0.232   0.062   6.045           15
JPN           0.332       0.280   0.053   4.670            9
RUS           0.131       0.087   0.045  23.263           44
CAN           0.208       0.173   0.035   6.592           23
CZE           0.293       0.261   0.032   2.086           16
NLD           0.342       0.311   0.031   3.097            8

Países de referencia:
         omega_real  omega_nulo  exceso       z  puesto_real  puesto_exceso
coun